In [8]:
import pyspark.sql.functions as F

from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DoubleType,
)

In [2]:
spark = SparkSession.builder.appName("DataFrameFilteringPractice").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/19 18:51:10 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/19 18:51:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 18:51:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer_name", StringType(), False),
        StructField("order_date", DateType(), False),
        StructField(
            "items",
            StructType(
                [
                    StructField("product_name", StringType(), False),
                    StructField("quantity", IntegerType(), False),
                    StructField("price", DoubleType(), False),
                ]
            ),
            False,
        ),
    ]
)

In [4]:
data = [
    (
        1,
        "Иван Петров",
        datetime.strptime("2023-09-15", "%Y-%m-%d").date(),
        ("Ноутбук", 1, 999.99),
    ),
    (
        2,
        "Анна Сидорова",
        datetime.strptime("2023-09-16", "%Y-%m-%d").date(),
        ("Смартфон", 2, 599.50),
    ),
    (
        3,
        "Елена Иванова",
        datetime.strptime("2023-09-17", "%Y-%m-%d").date(),
        ("Наушники", 3, 79.99),
    ),
    (
        4,
        "Анна Козлова",
        datetime.strptime("2023-09-18", "%Y-%m-%d").date(),
        ("Смартфон", 1, 699.99),
    ),
    (
        5,
        "Петр Смирнов",
        datetime.strptime("2023-09-19", "%Y-%m-%d").date(),
        ("Планшет", 1, 449.99),
    ),
]


In [5]:
df = spark.createDataFrame(data, schema)

In [6]:
df.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
+--------+-------------+----------+--------------------+



In [ ]:
df_sorted_by_date = df.sort(F.col("order_date").desc())
df_sorted_by_date.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
+--------+-------------+----------+--------------------+



In [ ]:
df_sorted_by_price = df.sort(F.col("items.price").asc())
df_sorted_by_price.show(truncate=False)

+--------+-------------+----------+---------------------+
|order_id|customer_name|order_date|items                |
+--------+-------------+----------+---------------------+
|3       |Елена Иванова|2023-09-17|{Наушники, 3, 79.99} |
|5       |Петр Смирнов |2023-09-19|{Планшет, 1, 449.99} |
|2       |Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5} |
|4       |Анна Козлова |2023-09-18|{Смартфон, 1, 699.99}|
|1       |Иван Петров  |2023-09-15|{Ноутбук, 1, 999.99} |
+--------+-------------+----------+---------------------+



In [17]:
df_sorted_by_name_quantity = df.orderBy(F.col("customer_name").asc(),
                                        F.col("items.quantity").desc())
df_sorted_by_name_quantity.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
+--------+-------------+----------+--------------------+



In [29]:
df_sorted_by_total_cost = df. \
            withColumn("total", F.round((F.col("items.quantity") * F.col("items.price")), 2)). \
            sort(F.desc("total"))
df_sorted_by_total_cost.show()

AssertionError: 

In [25]:
df_sorted_by_product_date = df.sort(F.col("items.product_name").asc(),
                                    F.col("order_date").desc())
df_sorted_by_product_date.show()

+--------+-------------+----------+--------------------+
|order_id|customer_name|order_date|               items|
+--------+-------------+----------+--------------------+
|       3|Елена Иванова|2023-09-17|{Наушники, 3, 79.99}|
|       1|  Иван Петров|2023-09-15|{Ноутбук, 1, 999.99}|
|       5| Петр Смирнов|2023-09-19|{Планшет, 1, 449.99}|
|       4| Анна Козлова|2023-09-18|{Смартфон, 1, 699...|
|       2|Анна Сидорова|2023-09-16|{Смартфон, 2, 599.5}|
+--------+-------------+----------+--------------------+



In [26]:
spark.stop()